# <center>Organización de Datos</center>
#### <center>Cátedra Ing. Rodriguez, Juan Manuel </center>

## <center>Feature Engineering</center>
### <center>Práctica Valores Faltantes</center>

> **Versión corregida y ampliada.** Cambios respecto de la original:
> - Se agregó la distinción **MCAR / MAR / MNAR**, que faltaba y es la base teórica para decidir *qué* estrategia de imputación tiene sentido.
> - El chequeo de "caracteres raros" ahora revisa varias representaciones de nulo, no solo `"-"`.
> - Se reemplazó el valor mágico `99` en `bathrooms` por un enfoque más seguro: imputar + agregar un indicador binario de "era faltante".
> - Se agregó **train/test split antes de fitear los imputadores** (KNN, Iterative/MICE), igual que en la notebook de transformación.
> - Se agregó un control de sanidad sobre los valores imputados por MICE (no deberían dar superficies o baños negativos).
> - Se resolvió el ejercicio de `show_strategies` con un ejemplo completo (imputación + comparación de distribuciones).

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import sklearn.preprocessing as skp

import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

pd.options.display.max_columns = None

#### Preparación del Dataset

In [ ]:
df = pd.read_csv("/content/ar_properties.csv")
df.head()

#### Verificamos la calidad de los datos

Como vimos en la teórica, los valores nulos pueden tener distintas formas de ser representados:

* `NaN`
* vacíos `""`
* algún carácter especial: `"-"`, `"?"`, `"NULL"`, `"N/A"`, `"none"`
* valores que no tienen sentido dado el significado de la variable (ej: superficie = -1)

La versión original de esta notebook solo chequeaba el carácter `"-"`. Acá revisamos varias representaciones a la vez, para que el ejemplo sea consistente con lo que se explica en el texto.

In [ ]:
representaciones_de_nulo = ["-", "?", "NULL", "null", "N/A", "n/a", "none", "None", ""]

for valor in representaciones_de_nulo:
    columnas_con_valor = df.astype("str").eq(valor).any(axis=0)
    columnas_afectadas = columnas_con_valor[columnas_con_valor].index.tolist()
    if columnas_afectadas:
        print(f"Valor '{valor}' encontrado en columnas: {columnas_afectadas}")

Chequeemos también variables numéricas cuyos valores no tienen sentido dado su significado (por ejemplo, un precio menor o igual a 0).

In [ ]:
columnas_con_numeros = ["price"]
(df[columnas_con_numeros] <= 0).any().to_frame("Menor o igual que 0 ?")

In [ ]:
df[df.price == 0].shape, df[df.price < 0].shape

In [ ]:
filas_totales = df.shape[0]
print(df.isna().sum() / filas_totales * 100)

### ¿Por qué falta el dato? MCAR, MAR y MNAR (esto faltaba en la versión original)

Antes de elegir *cómo* imputar, conviene preguntarse *por qué* falta el dato — porque de eso depende si una estrategia es válida o si va a introducir sesgo. Hay tres categorías clásicas:

* **MCAR (Missing Completely At Random):** la ausencia no depende de ningún valor, observado o no. Ejemplo: un error aleatorio de carga que pudo pasarle a cualquier fila con la misma probabilidad. Es el caso "más seguro": casi cualquier método de imputación (media, mediana, KNN) no introduce sesgo sistemático.
* **MAR (Missing At Random):** la ausencia depende de *otras variables observadas*, pero no del propio valor faltante. Ejemplo: `bedrooms` falta más seguido en avisos de tipo `Lote` o `Cochera` (porque no aplica tener dormitorios), y eso lo podemos ver mirando `property_type`. Acá, imputar usando otras columnas (KNN, MICE) tiene sentido; imputar con una constante global puede introducir sesgo.
* **MNAR (Missing Not At Random):** la ausencia depende del *propio valor faltante*, incluso después de controlar por lo observado. Ejemplo: si los avisos con superficies mucho más grandes que el promedio tienden a no publicar la superficie exacta (para no "asustar" con el precio implícito), la ausencia está relacionada con el valor que falta. Este es el caso más difícil: ninguna técnica de imputación basada en las demás columnas lo resuelve del todo, porque falta justamente la información que explicaría el patrón.

**TODO:** para `bedrooms`, `l4`, `l5` y `l6` de este dataset, propongan una hipótesis de a cuál de las tres categorías pertenece la ausencia, y qué evidencia buscarían en los datos para confirmarla (por ejemplo, cruzar el porcentaje de nulos por `property_type`).

In [ ]:
# Evidencia rápida para la hipótesis MAR de bedrooms: ¿el % de nulos varía según property_type?
(df.groupby("property_type")["bedrooms"].apply(lambda s: s.isna().mean() * 100)
   .sort_values(ascending=False)
   .to_frame("% nulos en bedrooms"))

#### Duplicados

Dependiendo del dataset, a veces tenemos información duplicada que no queremos.

- `df.duplicated()` → devuelve una serie de booleanos indicando si una fila es duplicada o no.
- `df.drop_duplicates()` → devuelve un DataFrame nuevo con las filas duplicadas eliminadas.

Nota: se le puede pasar el parámetro `subset=` para considerar solo algunas columnas al definir si está duplicado o no.

**Antes de borrar sin más:** conviene revisar si los duplicados se concentran en algún subgrupo (por ejemplo, un barrio o una inmobiliaria en particular). Si es así, eliminarlos "a ciegas" podría sesgar la muestra final en contra de ese subgrupo.

In [ ]:
df[df.duplicated(keep=False)]

In [ ]:
# Antes de eliminar, miramos si los duplicados se concentran en algún barrio
duplicados = df[df.duplicated(keep=False)]
if len(duplicados) > 0:
    display(duplicados["l3"].value_counts(normalize=True).head(10).to_frame("% de los duplicados"))

In [ ]:
size_antes = len(df)
df_filtrado = df.drop_duplicates()
size_despues = len(df_filtrado)
print(f"Se eliminaron: {size_antes - size_despues} filas duplicadas")

Nota: a veces es útil "resetear" el índice después de eliminar filas (ya sea por drop_duplicates o por algún otro filtro).

In [ ]:
df_filtrado.reset_index(drop=True, inplace=True)

Una vez analizados los datos, tenemos que tomar decisiones sobre qué hacer con los datos faltantes.

#### Opción 0: Eliminarlos del dataset

Vamos a trabajar con una versión reducida del dataset, seleccionando solo algunas columnas.

In [ ]:
df_filtrado = df.copy()

cond_lugar = df_filtrado["l2"] == "Capital Federal"
cond_moneda = df_filtrado["currency"] == "USD"
cond_operacion = df_filtrado["operation_type"] == "Venta"

df_filtrado = df_filtrado[cond_lugar & cond_moneda & cond_operacion]

columnas = {"l2": "ciudad", "l3": "barrio"}
df_filtrado.rename(columns=columnas, inplace=True)

df_filtrado.head()

In [ ]:
df_eliminar_nans = df_filtrado.copy()

filas_totales = df_eliminar_nans.shape[0]
print(df_eliminar_nans.isna().sum() / filas_totales * 100)

Podemos pensar en eliminar las siguientes columnas, que tienen una proporción alta de nulos.

In [ ]:
columnas_eliminar_NANs = ["bedrooms", "l4", "l5", "l6"]
df_eliminar_nans.drop(columnas_eliminar_NANs, axis="columns", inplace=True)

filas_totales = df_eliminar_nans.shape[0]
print(df_eliminar_nans.isna().sum() / filas_totales * 100)

Eliminamos el resto de las filas con NaNs.

In [ ]:
df_eliminar_nans.dropna(inplace=True)

print("cantidad de registros originales: " + str(df_filtrado.shape[0]))
print("cantidad de registros finales: " + str(df_eliminar_nans.shape[0]))

porcentaje = (df_filtrado.shape[0] - df_eliminar_nans.shape[0]) / df_filtrado.shape[0] * 100
print("Eliminamos el " + str(porcentaje) + " % de los registros")

#### Opción 1: Tratarla como una "categoría" o valor más

**Corrección importante respecto de la versión original:** ahí se rellenaba `bathrooms` (una variable numérica continua) con el valor `99`, tratándolo como si fuera "una categoría más". El problema es que `99` sigue siendo un número dentro de una columna numérica: cualquier media, desvío, distancia (KNN) o modelo lineal que use esa columna va a tratar a esos `99` como valores reales, distorsionando fuertemente la variable.

Una alternativa más segura para variables numéricas: imputar con un valor razonable (mediana, por ejemplo) y agregar una **columna indicadora binaria** de "este valor era faltante". Así el modelo puede aprender que esas filas son distintas, sin que el valor numérico en sí quede distorsionado. Para variables categóricas, en cambio, sí tiene sentido usar directamente una categoría nueva como `"Unknown"` (ahí no hay una escala numérica que se rompa).

In [ ]:
df_completar_con_valor1 = df_filtrado.copy()

# Indicador de que el valor era faltante (antes de imputar)
df_completar_con_valor1["bathrooms_era_nan"] = df_completar_con_valor1["bathrooms"].isna()

# Imputamos con la mediana en vez de un valor arbitrario como 99
df_completar_con_valor1["bathrooms"] = df_completar_con_valor1["bathrooms"].fillna(
    df_completar_con_valor1["bathrooms"].median()
)

# Para una variable categórica sí tiene sentido una categoría nueva
df_completar_con_valor1["barrio"] = df_completar_con_valor1["barrio"].fillna("Unknown")

df_completar_con_valor1[["bathrooms", "bathrooms_era_nan", "barrio"]].head()

#### Opción 2: completar usando info de esa misma columna (univariadas)

Completar con la mediana, promedio, moda o una constante.

In [ ]:
df_completar_con_valor2 = df_filtrado.copy()

# Devuelve el valor de imputación de las tres estrategias para esa columna
def show_strategies(df, name_col, k=99):
    _df = df[[name_col]].copy()
    s = df[name_col]

    _df["mediana"] = s.fillna(s.median())
    _df["media"] = s.fillna(s.mean())
    _df["moda"] = s.fillna(s.mode()[0])
    _df["constante"] = k

    return _df[s.isna()]

show_strategies(df_completar_con_valor2, "rooms", 99).head()

### Resolución del ejercicio: comparar las distribuciones antes/después de imputar `rooms`

Esto quedaba como tarea sin resolver en la versión original. Acá va un ejemplo completo, comparando cómo cambia la distribución de `rooms` según la estrategia de imputación elegida (excluimos la opción "constante=99" del gráfico a propósito, porque distorsiona tanto la escala que no se puede comparar visualmente con las demás — que es, de nuevo, la misma lección que vimos con `bathrooms`).

In [ ]:
serie_rooms = df_completar_con_valor2["rooms"]

rooms_mediana = serie_rooms.fillna(serie_rooms.median())
rooms_media = serie_rooms.fillna(serie_rooms.mean())
rooms_moda = serie_rooms.fillna(serie_rooms.mode()[0])

fig, ax = plt.subplots(figsize=(8, 5))
serie_rooms.dropna().plot(kind="kde", label="Original (sin NaNs)", ax=ax)
rooms_mediana.plot(kind="kde", label="Imputado con mediana", ax=ax)
rooms_media.plot(kind="kde", label="Imputado con media", ax=ax)
rooms_moda.plot(kind="kde", label="Imputado con moda", ax=ax)

plt.title("Distribución de 'rooms' antes y después de imputar (univariado)")
plt.xlabel("rooms")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

Noten que las tres estrategias univariadas "aplastan" la distribución alrededor de un único valor (la moda es la que más lo hace, porque concentra toda la masa faltante en un solo punto) — es la limitación central de imputar usando solo la propia columna, y la motivación para pasar a estrategias multivariadas.

**TODO:** implementen el mismo ejemplo con `SimpleImputer` de scikit-learn (`strategy='median'`, `'mean'`, `'most_frequent'`) en vez de `fillna`, fiteando sobre train y aplicando sobre test.

#### Opción 3: completar usando info de las demás columnas (multivariada)

Igual que en la notebook de transformación de datos, acá también hay que fitear el imputador solo con datos de entrenamiento.

In [ ]:
from sklearn.model_selection import train_test_split

# Variables numéricas que vamos a comparar en ambos métodos
columnas = ["surface_total", "surface_covered", "bathrooms", "rooms"]

df_base = df_filtrado[columnas].copy()

# Split ANTES de fitear cualquier imputador (esto faltaba en la versión original)
df_base_train, df_base_test = train_test_split(df_base, test_size=0.2, random_state=42)

print("Train:", df_base_train.shape, "Test:", df_base_test.shape)

##### Aproximación tipo MICE (`IterativeImputer`)

In [ ]:
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression

# Definimos el modelo base (se usa para predecir valores faltantes)
lr = LinearRegression()

# Creamos el imputador iterativo
imputador_iterativo = IterativeImputer(
    estimator=lr,
    missing_values=np.nan,
    max_iter=20,
    verbose=0,
    random_state=0,
)

# Ajustamos SOLO con train...
imputador_iterativo.fit(df_base_train)

# ...y aplicamos (transform) a train y test
df_mice_train = pd.DataFrame(
    imputador_iterativo.transform(df_base_train), columns=columnas, index=df_base_train.index
)
df_mice_test = pd.DataFrame(
    imputador_iterativo.transform(df_base_test), columns=columnas, index=df_base_test.index
)

df_mice_train.head()

### Control de sanidad sobre los valores imputados (esto faltaba en la versión original)

`IterativeImputer` con una regresión lineal como estimador puede **extrapolar** fuera del rango razonable de la variable — por ejemplo, predecir una superficie o una cantidad de baños negativa, que no tiene sentido físico. Antes de dar por buena la imputación, conviene chequear esto explícitamente.

In [ ]:
columnas_no_negativas = ["surface_total", "surface_covered", "bathrooms", "rooms"]

valores_invalidos = (df_mice_train[columnas_no_negativas] < 0).sum()
print("Valores negativos introducidos por MICE (deberían ser 0):")
print(valores_invalidos)

if valores_invalidos.sum() > 0:
    print("\n Hay valores negativos: en un caso real, conviene clipear a 0 o a un mínimo razonable, ")
    print("   o usar un estimador no lineal (ej. RandomForestRegressor) que no extrapola de la misma forma.")

##### KNN Imputer

In [ ]:
from sklearn.impute import KNNImputer

# KNNImputer trabaja con distancias → necesita números
imputador_knn = KNNImputer(
    n_neighbors=2,
    add_indicator=False,  # si es True, agrega una columna extra "era faltante" por cada variable imputada
)

# Ajustamos SOLO con train
imputador_knn.fit(df_base_train)

df_knn_train = pd.DataFrame(
    imputador_knn.transform(df_base_train), columns=columnas, index=df_base_train.index
)
df_knn_test = pd.DataFrame(
    imputador_knn.transform(df_base_test), columns=columnas, index=df_base_test.index
)

df_knn_train.head()

##### Comparativa

Nota: `df_base_train.isna().sum()` ya no debería mostrar NaNs remanentes después de `KNNImputer` o `IterativeImputer`.

In [ ]:
print("Valores faltantes antes de imputar (train):")
print(df_base_train.isna().sum())

print("\nValores faltantes luego de KNN (train):")
print(df_knn_train.isna().sum())

print("\nValores faltantes luego de IterativeImputer (train):")
print(df_mice_train.isna().sum())

In [ ]:
import matplotlib.pyplot as plt

# Percentiles para recortar, calculados sobre TRAIN
p_inf = 0.01
p_sup = 0.99

for columna in columnas:
    plt.figure(figsize=(8, 5))

    lower = df_base_train[columna].quantile(p_inf)
    upper = df_base_train[columna].quantile(p_sup)

    base_filtrado = df_base_train[columna].dropna()
    base_filtrado = base_filtrado[(base_filtrado >= lower) & (base_filtrado <= upper)]

    knn_filtrado = df_knn_train[columna]
    knn_filtrado = knn_filtrado[(knn_filtrado >= lower) & (knn_filtrado <= upper)]

    mice_filtrado = df_mice_train[columna]
    mice_filtrado = mice_filtrado[(mice_filtrado >= lower) & (mice_filtrado <= upper)]

    base_filtrado.plot(kind="kde", label="Original")
    knn_filtrado.plot(kind="kde", label="KNN")
    mice_filtrado.plot(kind="kde", label="Iterative")

    plt.title(f"Distribución (recortada) - {columna} (train)")
    plt.xlabel(columna)
    plt.ylabel("Densidad")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

### TODO final

1. Repitan la comparativa de KNN vs Iterative, pero evaluando sobre **test** (`df_knn_test`, `df_mice_test`) en vez de train — ¿las distribuciones imputadas en test se parecen a las de train? Es una forma indirecta de chequear que el imputador generaliza bien.
2. Prueben `add_indicator=True` en `KNNImputer` y describan qué columnas nuevas aparecen.
3. Vuelvan a la pregunta de MCAR/MAR/MNAR del principio: para `rooms`, ¿la imputación multivariada (KNN/MICE) tiene más sentido que la univariada (mediana/media)? Justifiquen con lo que vieron en el gráfico de `groupby("property_type")`.